
# [Step 1] 라이브러리 설치, 폴더 세팅 및 100종목 데이터 수집


In [2]:
# 1. 필수 라이브러리 설치 및 임포트

!pip install yfinance pandas numpy -q

import os
import time
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [3]:
# 2. 프로젝트 폴더 세팅 (코랩 로컬 디렉토리 구조)
BASE_DIR = Path(".")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_META = BASE_DIR / "data" / "meta"
RESULTS_FIGURES = BASE_DIR / "results" / "figures"
RESULTS_TABLES = BASE_DIR / "results" / "tables"

for path in [DATA_RAW, DATA_PROCESSED, DATA_META, RESULTS_FIGURES, RESULTS_TABLES]:
    path.mkdir(parents=True, exist_ok=True)

print("[+] 코랩 데이터 디렉토리 세팅 완료!")

[+] 코랩 데이터 디렉토리 세팅 완료!


In [4]:
# 3. 수집 기간 정의 (어제 기준 2년 6개월 역산)
end_date = datetime.today() - timedelta(days=1)
start_date = end_date - timedelta(days=30 * 30)
start_str = start_date.strftime("%Y-%m-%d")
end_str = end_date.strftime("%Y-%m-%d")

In [5]:
# 4. 대형주(35), 중형주(35), 소형주(30) 종목 정의
# KRX(한국거래소) 공식 시가총액 기준
large_caps = ["005930.KS", "000660.KS", "073540.KS", "207940.KS", "005490.KS", "005380.KS", "035420.KS", "051910.KS", "035720.KS", "006400.KS", "012330.KS", "068270.KS", "105560.KS", "055550.KS", "015760.KS", "000270.KS", "017670.KS", "096770.KS", "032830.KS", "003550.KS", "033780.KS", "000810.KS", "018260.KS", "086790.KS", "010950.KS", "009150.KS", "051900.KS", "034730.KS", "251270.KS", "011170.KS", "009830.KS", "034220.KS", "010130.KS", "004020.KS", "016360.KS"]
mid_caps = ["042660.KS", "078930.KS", "000720.KS", "028260.KS", "036570.KS", "009240.KS", "008930.KS", "000100.KS", "010620.KS", "069960.KS", "001040.KS", "004990.KS", "021240.KS", "005940.KS", "006260.KS", "192820.KS", "161390.KS", "285130.KS", "023530.KS", "007070.KS", "047040.KS", "001450.KS", "000120.KS", "079160.KS", "003240.KS", "064350.KS", "004370.KS", "052690.KS", "011210.KS", "020150.KS", "001800.KS", "000080.KS", "005830.KS", "006360.KS", "003490.KS"]
small_caps = ["083660.KQ", "044180.KQ", "011040.KQ", "021040.KQ", "054340.KQ", "053030.KQ", "032960.KQ", "038340.KQ", "037350.KQ", "115530.KQ", "023160.KQ", "024840.KQ", "036480.KQ", "039560.KQ", "058450.KQ", "065560.KQ", "101330.KQ", "054050.KQ", "036640.KQ", "041510.KQ", "025950.KQ", "060230.KQ", "060250.KQ", "035620.KQ", "024810.KQ", "038110.KQ", "043260.KQ", "065620.KQ", "052420.KQ", "033600.KQ"]
all_tickers = large_caps + mid_caps + small_caps
ticker_group_map = {t: "Large" for t in large_caps}
ticker_group_map.update({t: "Medium" for t in mid_caps})
ticker_group_map.update({t: "Small" for t in small_caps})

In [6]:
# 5. 루프를 돌며 종목별 데이터 다운로드 및 병합
all_frames = []
drop_count = 0
for idx, ticker in enumerate(all_tickers, start=1):
    group = ticker_group_map[ticker]
    try:
        df = yf.download(ticker, start=start_str, end=end_str, progress=False, auto_adjust=False)
        if df.empty or len(df) < 10:
            drop_count += 1
            continue
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        df = df.reset_index().rename(columns={"Date": "date", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Adj Close": "adj_close", "Volume": "volume"})
        df["ticker"] = ticker
        df["group"] = group
        all_frames.append(df)
        time.sleep(0.1)
    except Exception as e: drop_count += 1

In [7]:
# 6. 전체 수집 데이터 하나의 데이터프레임으로 통합
if all_frames:
    raw_total_df = pd.concat(all_frames, ignore_index=True)
    raw_total_df["date"] = pd.to_datetime(raw_total_df["date"])
    raw_total_df = raw_total_df.sort_values(["ticker", "date"]).reset_index(drop=True)
    raw_total_df.to_csv(DATA_RAW / "raw_ohlcv_100.csv", index=False, encoding="utf-8-sig")
    print(f"[+] 수집 완료: {len(raw_total_df)}행")